# Preprocessing and Feature Engineering with Numerical Data

This notebook explains:
- Data preprocessing
- Feature engineering
- Handling missing values
- Feature scaling
- Normalization
- Outlier detection
- Feature creation
- Binning
- Polynomial features
- Correlation analysis

All examples use **numerical real-world style datasets**.



## Real-World Scenario

Suppose we are building a machine learning model to predict **house prices**.

The dataset contains only numerical features such as:
- House size
- Number of rooms
- Age of house
- Distance to city center
- Crime rate
- School rating
- House price

We will preprocess the data and engineer better features before training a model.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

np.random.seed(42)


In [3]:

# Create a synthetic real-world numerical dataset

n = 100

data = pd.DataFrame({
    'house_size': np.random.normal(1800, 400, n),
    'num_rooms': np.random.randint(2, 8, n),
    'house_age': np.random.randint(1, 40, n),
    'distance_city': np.random.normal(15, 5, n),
    'crime_rate': np.random.uniform(1, 10, n),
    'school_rating': np.random.uniform(1, 10, n)
})

# Create target variable
data['house_price'] = (
    data['house_size'] * 120 +
    data['num_rooms'] * 15000 -
    data['house_age'] * 1000 -
    data['distance_city'] * 2500 -
    data['crime_rate'] * 5000 +
    data['school_rating'] * 8000 +
    np.random.normal(0, 10000, n)
)

# Add missing values
data.loc[5, 'house_size'] = np.nan
data.loc[12, 'crime_rate'] = np.nan
data.loc[30, 'distance_city'] = np.nan

data.head()


,house_size,num_rooms,house_age,distance_city,crime_rate,school_rating,house_price
0,2074.732716,6,36,14.036101,3.148210,1.011087,256130.761560
1,864.162799,4,38,14.229816,9.370813,6.384945,95116.653199
2,884.375497,3,34,21.784284,1.952507,6.465758,120314.970152
3,2372.992767,3,17,13.202814,7.290917,3.227860,253685.013436
4,2285.888824,4,37,21.783167,2.702994,6.025714,290708.920356



# 1. Handling Missing Values

Real-world datasets often contain missing values.

Examples:
- Sensor failure
- User skipped information
- Corrupted database entries

We will replace missing values using the column mean.


In [ ]:

# Check missing values

data.isnull().sum()


In [ ]:

# Fill missing values using mean

imputer = SimpleImputer(strategy='mean')

data_imputed = pd.DataFrame(
    imputer.fit_transform(data),
    columns=data.columns
)

data_imputed.head()



# 2. Feature Scaling

Different features have different ranges.

Examples:
- house_size → 1000 to 3000
- crime_rate → 1 to 10

Machine learning models perform better when features are scaled.


In [ ]:

# Standardization

features = data_imputed.drop('house_price', axis=1)

scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

scaled_df = pd.DataFrame(
    scaled_features,
    columns=features.columns
)

scaled_df.head()



# 3. Normalization using Min-Max Scaling

Min-Max scaling converts values to a range between 0 and 1.


In [ ]:

minmax = MinMaxScaler()

normalized_features = minmax.fit_transform(features)

normalized_df = pd.DataFrame(
    normalized_features,
    columns=features.columns
)

normalized_df.head()



# 4. Outlier Detection

Outliers are unusually large or small values.

Examples:
- Extremely expensive house
- Extremely large house

We can detect outliers using boxplots.


In [ ]:

plt.figure(figsize=(8,4))
plt.boxplot(data_imputed['house_size'])
plt.title('Boxplot of House Size')
plt.ylabel('Square Feet')
plt.show()



# 5. Feature Engineering

Feature engineering means creating new useful features from existing data.

Examples:
- price_per_room
- age_per_room
- luxury_score


In [ ]:

# Create new engineered features

data_imputed['size_per_room'] = (
    data_imputed['house_size'] / data_imputed['num_rooms']
)

data_imputed['location_score'] = (
    data_imputed['school_rating'] / data_imputed['crime_rate']
)

data_imputed[['size_per_room', 'location_score']].head()



# 6. Feature Binning

Binning groups continuous numerical values into ranges.

Example:
- Young houses
- Medium age houses
- Old houses


In [ ]:

# Create age categories

data_imputed['age_category'] = pd.cut(
    data_imputed['house_age'],
    bins=[0, 10, 25, 40],
    labels=[1, 2, 3]
)

data_imputed[['house_age', 'age_category']].head()



# 7. Polynomial Features

Polynomial features help models learn non-linear relationships.

Example:
- house_size²
- house_size × num_rooms


In [ ]:

poly = PolynomialFeatures(degree=2, include_bias=False)

poly_features = poly.fit_transform(
    data_imputed[['house_size', 'num_rooms']]
)

poly_feature_names = poly.get_feature_names_out(
    ['house_size', 'num_rooms']
)

poly_df = pd.DataFrame(
    poly_features,
    columns=poly_feature_names
)

poly_df.head()



# 8. Correlation Analysis

Correlation helps identify relationships between features.


In [ ]:

correlation_matrix = data_imputed.corr(numeric_only=True)

correlation_matrix


In [ ]:

plt.figure(figsize=(10,6))

plt.imshow(correlation_matrix, cmap='coolwarm')
plt.colorbar()

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)

plt.title('Correlation Matrix')
plt.show()



# 9. Train-Test Split

Machine learning datasets are usually divided into:
- Training data
- Testing data


In [ ]:

X = data_imputed.drop('house_price', axis=1)
y = data_imputed['house_price']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)



# Summary

In this notebook, we learned:

## Preprocessing
- Handling missing values
- Standardization
- Normalization
- Outlier detection

## Feature Engineering
- Creating new features
- Feature binning
- Polynomial features
- Correlation analysis

These steps improve machine learning model performance by preparing better quality input data.
